<a href="https://colab.research.google.com/github/FathimaNazarudeen/medical/blob/main/4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
data=pd.read_csv("/content/thirddatasetdata2.csv")

In [ ]:
data['Label'].value_counts()

In [ ]:
data.columns

**DATA PREPROCESSING**

In [ ]:
data_real = data.replace(np.inf, np.nan)

In [ ]:
data_real.isnull().sum().sum()

In [ ]:
data_df = data_real.dropna(axis=0)

In [ ]:
data_df.isnull().sum().sum()

In [ ]:
data_df

In [ ]:
data_X = data_df.drop(['Label'], axis = 1)

In [ ]:
data_X.columns 

In [ ]:
data_df.isnull().sum().sum()

In [ ]:
data_y = data_df['Label']

**Label Encoding**

In [ ]:
from sklearn.preprocessing import LabelEncoder 

In [ ]:
le = LabelEncoder()

In [ ]:
data_y_trans = le.fit_transform(data_y)
le_fid = LabelEncoder()

**Train Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train, y_test = train_test_split(data_X, data_y, train_size = 0.7)

**Standardization**

In [ ]:
from sklearn.preprocessing import StandardScaler 
ss = StandardScaler()
X_train_std = ss.fit_transform(X_train)
X_test_std = ss.fit_transform(X_test)

**Feature Selection**

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
model = ExtraTreesClassifier(bootstrap=False, ccp_alpha=0.0, class_weight=None,
                     criterion='gini', max_depth=None, max_features='auto',
                     max_leaf_nodes=None, max_samples=None,
                     min_impurity_decrease=0.0, 
                     min_samples_leaf=1, min_samples_split=2,
                     min_weight_fraction_leaf=0.0, n_estimators=20,
                     n_jobs=None, oob_score=False, random_state=42, verbose=0,
                     warm_start=False)

model.fit(data_X, data_y_trans)

In [ ]:
model.feature_importances_

In [ ]:
feature_importance_std = pd.Series(model.feature_importances_, index=data_X.columns)
feature_importance_std.nlargest(20).plot(kind='bar', title='Standardised Dataset Feature Selection using ExtraTreesClassifier')

In [ ]:
data_new_20features_X = data_X[['imaging.radiomics.Shape_sphericity', 'imaging.radiomics.Shape_spherDisprop' ,'imaging.radiomics.Shape_orig_spherDisprop', 'imaging.radiomics.Shape_compactness2','imaging.radiomics.Wavelet_LHH_stats_max', 'imaging.radiomics.LoG_sigma_3_mm_2D_stats_skewness', 'imaging.radiomics.Wavelet_LHH_glcm_correl1', 'imaging.radiomics.Shape_orig_maxDiameter2Dy', 'imaging.radiomics.Wavelet_HHL_stats_kurtosis', 'imaging.radiomics.Wavelet_LHL_glcm_invDiffnorm', 'imaging.radiomics.Shape_orig_maxDiameter3D', 'imaging.radiomics.LoG_sigma_2_5_mm_3D_stats_skewness', 'imaging.radiomics.Wavelet_HLH_glcm_clusProm', 'imaging.radiomics.Stats_min', 'imaging.radiomics.Wavelet_HHH_glcm_clusShade', 'imaging.radiomics.LoG_sigma_4_mm_2D_stats_skewness', 'imaging.radiomics.Shape_compactness2', 'imaging.radiomics.LoG_sigma_5_mm_3D_stats_uniformityPos' ,'imaging.radiomics.LoG_sigma_4_mm_3D_stats_kurtosis', 'imaging.radiomics.Wavelet_LHL_stats_uniformity']]

In [ ]:
data_new_20features_X

**Train Test Split 20 Feature Dataset**

In [ ]:
from sklearn.model_selection import train_test_split
X_train_20, X_test_20, y_train_20, y_test_20 = train_test_split(data_new_20features_X, data_y_trans, train_size = 0.7)

**Standardization of the 20 Feature Dataset**

In [ ]:
from sklearn.preprocessing import StandardScaler 
ss_20 = StandardScaler()
X_train_std_20 = ss_20.fit_transform(X_train_20)
X_test_std_20 = ss_20.fit_transform(X_test_20)

**CLASSIFICATION**

**Random forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42, n_estimators=20, max_depth=90)
rf.fit(X_train_std_20, y_train_20)

In [ ]:
rf_y_pred = rf.predict(X_test_std_20)

In [ ]:
from sklearn.metrics import accuracy_score 

In [ ]:
acc_score = accuracy_score(y_test_20, rf_y_pred)
print("Accuracy Score for Random_Forest: \n", acc_score*100)

In [ ]:
from sklearn.metrics import classification_report 
from sklearn.metrics import confusion_matrix 

In [ ]:
print("Classification Report for Random Forest: \n", classification_report(y_test_20,rf_y_pred))

In [ ]:
rf_conf_mat = confusion_matrix(y_test_20, rf_y_pred)
print("Random Forest Confusion: \n", rf_conf_mat)

RoC curve Function

In [ ]:
def RoC_Curve(classifier, X_val, y_val, title): 
        """ RoC Curve for Classifier 
        Parameters: 
        ------------
        classifier: Machine Learning Classifier to be Evaluated
        X_val: Validation Dataset
        y_val: Label/Target of Validation Dataset

        Attributes:
        Plots the Graph    
        
        Note: Some part of this Method code is taken 
            from Sklearn Website
        """

        lw = 2
        n_classes = 3
        y_test1 = to_categorical(y_val)
        pred_RFC_proba = classifier.predict_proba(X_val)
        y_score = pred_RFC_proba

        # Compute ROC curve and ROC area for each class
        fpr = dict()
        tpr = dict()
        roc_auc = dict()
        for i in range(n_classes):
            fpr[i], tpr[i], _ = roc_curve(y_test1[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        # Compute micro-average ROC curve and ROC area
        fpr["micro"], tpr["micro"], _ = roc_curve(y_test1.ravel(), y_score.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        # First aggregate all false positive rates
        all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

        # Then interpolate all ROC curves at this points
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += interp(all_fpr, fpr[i], tpr[i])

        # Finally average it and compute AUC
        mean_tpr /= n_classes

        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr
        roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

        # Plot all ROC curves
        plt.figure(figsize=(20,10))
        plt.plot(fpr["micro"], tpr["micro"],
                label='micro-average ROC curve (area = {0:0.2f})'
                    ''.format(roc_auc["micro"]),
                color='deeppink', linestyle=':', linewidth=4)

        plt.plot(fpr["macro"], tpr["macro"],
                label='macro-average ROC curve (area = {0:0.2f})'
                    ''.format(roc_auc["macro"]),
                color='navy', linestyle=':', linewidth=4)

        list_class = ['adeno','squamous','other']
        for i in range(n_classes):
            plt.plot(fpr[i], tpr[i], lw=lw,
                    label='ROC curve of class {0} (area = {1:0.2f})'
                    ''.format(list_class[i], roc_auc[i]))

        plt.plot([0, 1], [0, 1], 'k--', lw=lw)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(title) 
        plt.legend(loc="lower right")
        plt.show()

In [ ]:
from keras.utils.np_utils import to_categorical
from sklearn.metrics import roc_curve, auc
from scipy import interp
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
title = 'Receiver operating characteristic of Random Forest'
RoC_Curve(rf, X_test_std_20, y_test_20, title)

**Decision tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
ml_dec = DecisionTreeClassifier()

In [ ]:
ml_dec.fit(X_train_std_20, y_train_20)

In [ ]:
res_dec=ml_dec.predict(X_test_std_20)

In [ ]:
accuracy_1=accuracy_score(y_test_20,res_dec)
print("Decision tree :",accuracy_1*100)

**XG Boost**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
gradient_boost = GradientBoostingClassifier()
gradient_boost.fit(X_train_std_20, y_train_20)

In [ ]:
res_xg=gradient_boost.predict(X_test_std_20)

In [ ]:
acc_score_xg = accuracy_score(y_test_20, res_xg)
print("Accuracy Score for GradientBoosting: \n", acc_score_xg*100)

**Naive Bayes**

In [ ]:
from sklearn.naive_bayes import GaussianNB 

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train_std_20, y_train_20)
gnb_y_pred = gnb.predict(X_test_std_20)

In [ ]:
acc_score_gnb = accuracy_score(y_test_20, gnb_y_pred)
print("Accuracy Score for Naive: \n", acc_score_gnb*100)